# Exam Grading System — Generic (any course)
Upload an answer key (PDF/DOCX) and student exam PDFs. The system extracts everything via Claude vision.

In [ ]:
!pip install -q anthropic python-docx pymupdf Pillow pandas
!apt-get install -y libreoffice > /dev/null 2>&1

import os, io, json, re, base64, textwrap, time, subprocess
from pathlib import Path
from datetime import datetime

import fitz  # PyMuPDF
from docx import Document
from PIL import Image
import pandas as pd
from anthropic import Anthropic, APIStatusError, APIConnectionError, RateLimitError
from google.colab import files, userdata

print("✅ All dependencies installed.")

## API key + retry helper

In [ ]:
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    ANTHROPIC_API_KEY = input("Enter your Anthropic API key: ").strip()

client = Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = "claude-sonnet-4-20250514"
print(f"✅ Anthropic client ready. Model: {MODEL}")


def call_claude_with_retry(max_retries: int = 6, base_delay: float = 2.0, **kwargs):
    """Wrapper around client.messages.create with exponential backoff
    on 529 (overloaded), 429 (rate limit), 5xx, and connection errors."""
    for attempt in range(max_retries):
        try:
            return client.messages.create(**kwargs)
        except (APIStatusError, RateLimitError, APIConnectionError) as e:
            status = getattr(e, "status_code", None)
            retryable = isinstance(e, (RateLimitError, APIConnectionError)) or \
                        (status is not None and (status == 529 or status == 429 or status >= 500))
            if not retryable or attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"    ⏳ API {status or 'error'}: retrying in {delay:.0f}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(delay)
    raise RuntimeError("Exhausted retries")

print("✅ Retry helper ready.")

## Upload answer key + student papers

In [ ]:
WORK_DIR = Path("exam_data")
ANSWERS_DIR = WORK_DIR / "answer_key"
STUDENTS_DIR = WORK_DIR / "student_papers"
OUTPUT_DIR = WORK_DIR / "output"
DRAWINGS_DIR = OUTPUT_DIR / "drawings"

for d in [ANSWERS_DIR, STUDENTS_DIR, OUTPUT_DIR, DRAWINGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("📂 Upload the ANSWER KEY (.pdf or .docx) file:")
uploaded_answer = files.upload()
for name, data in uploaded_answer.items():
    (ANSWERS_DIR / name).write_bytes(data)
    ANSWER_KEY_PATH = ANSWERS_DIR / name
    print(f"   ✅ {name}")

print("\n📂 Upload STUDENT EXAM PDFs (select all at once):")
uploaded_students = files.upload()
student_paths = []
for name, data in uploaded_students.items():
    path = STUDENTS_DIR / name
    path.write_bytes(data)
    student_paths.append(path)
    print(f"   ✅ {name}")

print(f"\n🎯 Answer key: {ANSWER_KEY_PATH.name}")
print(f"🎯 Student papers: {len(student_paths)} files")

## Answer type taxonomy (10 categories)

In [ ]:
ANSWER_TYPE_TAXONOMY = {
    "1":  {"name": "Selection-Based", "subtypes": {
        "1.1": "Single MCQ",
        "1.2": "Multi-select MCQ",
        "1.3": "True/False",
        "1.4": "True/False with correction",
        "1.5": "Matching",
        "1.6": "Ordering",
        "1.7": "Fill-in-the-blank with word bank"
    }},
    "2":  {"name": "Recall and Short Answer", "subtypes": {
        "2.1": "Single term answer",
        "2.2": "Definition",
        "2.3": "List/Enumerate",
        "2.4": "List with explanation",
        "2.5": "Differentiation/Comparison",
        "2.6": "Fill-in-the-blank (no bank)",
        "2.7": "Acronym expansion",
        "2.8": "Give an example"
    }},
    "3":  {"name": "Mathematical and Computational", "subtypes": {
        "3.1": "Formula application",
        "3.2": "Multi-step calculation",
        "3.3": "Conversion",
        "3.14": "Scheduling calculations"
    }},
    "4":  {"name": "Code and Programming", "subtypes": {
        "4.1": "Write code snippet",
        "4.2": "Complete missing code",
        "4.3": "Debug/fix code"
    }},
    "5":  {"name": "Diagrammatic and Visual", "subtypes": {
        "5.1": "Flowchart",
        "5.2": "ER diagram",
        "5.3": "UML diagram",
        "5.9": "Network Diagram (AOA/AON)"
    }},
    "6":  {"name": "Table and Structured Data", "subtypes": {
        "6.1": "Fill-in-the-table",
        "6.2": "Build a table from scratch"
    }},
    "7":  {"name": "Algorithm Execution and Tracing", "subtypes": {
        "7.1": "Trace algorithm execution",
        "7.2": "Show data-structure state"
    }},
    "8":  {"name": "Analysis and Interpretation", "subtypes": {
        "8.1": "Interpret output/results",
        "8.2": "Compare scenarios"
    }},
    "9":  {"name": "Design and Open-Ended", "subtypes": {
        "9.1": "Design a system/schema",
        "9.2": "Propose a solution"
    }},
    "10": {"name": "Essay and Explanation", "subtypes": {
        "10.1": "Explain a concept",
        "10.2": "Discuss pros/cons"
    }}
}

# Readable listing for prompts (names only, no code numbers shown to LLM)
TAXONOMY_PROMPT_TEXT = ""
for cat in ANSWER_TYPE_TAXONOMY.values():
    TAXONOMY_PROMPT_TEXT += f"Category: {cat['name']}\n"
    for sub_name in cat['subtypes'].values():
        TAXONOMY_PROMPT_TEXT += f"  - {sub_name}\n"


def repair_json(text: str) -> str:
    """Fix common LLM JSON mistakes: unquoted trailing text, trailing commas."""
    lines = text.split("\n")
    repaired = []
    for line in lines:
        m = re.match(r'^(\s*"[^"]+"\s*:\s*)"(.+?)"(.+)$', line)
        if m:
            prefix, value, after = m.group(1), m.group(2), m.group(3)
            after_stripped = after.strip()
            if after_stripped and after_stripped[0] not in [",", "}", "]"]:
                terminator = ""
                if after_stripped[-1] in [",", "}", "]"]:
                    terminator = after_stripped[-1]
                    after_stripped = after_stripped[:-1]
                extra = after_stripped.replace('"', "'").strip()
                line = f'{prefix}"{value} {extra}"{terminator}'
        repaired.append(line)
    text = "\n".join(repaired)
    text = re.sub(r',\s*([}\]])', r'\1', text)
    return text

print("✅ Taxonomy + JSON repair ready.")

## PDF/DOCX → images

In [ ]:
def pdf_to_images(pdf_path: Path, dpi: int = 200) -> list:
    doc = fitz.open(str(pdf_path))
    images = []
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)
    for page_num in range(len(doc)):
        pix = doc[page_num].get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        images.append(img)
    doc.close()
    return images


def docx_to_images(docx_path: Path, dpi: int = 200) -> list:
    out_dir = docx_path.parent / "_docx_pdf_tmp"
    out_dir.mkdir(exist_ok=True)
    subprocess.run(
        ["libreoffice", "--headless", "--convert-to", "pdf",
         "--outdir", str(out_dir), str(docx_path)],
        check=True, capture_output=True
    )
    pdf_path = out_dir / (docx_path.stem + ".pdf")
    return pdf_to_images(pdf_path, dpi=dpi)


def answer_key_to_images(path: Path, dpi: int = 200) -> list:
    ext = path.suffix.lower()
    if ext == ".pdf":
        return pdf_to_images(path, dpi=dpi)
    elif ext == ".docx":
        return docx_to_images(path, dpi=dpi)
    else:
        raise ValueError(f"Unsupported answer key format: {ext}. Use PDF or DOCX.")


def image_to_base64(img: Image.Image, max_size: int = 1600) -> str:
    w, h = img.size
    if max(w, h) > max_size:
        ratio = max_size / max(w, h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    buffer = io.BytesIO()
    img.save(buffer, format="PNG", optimize=True)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

print("✅ Image conversion ready.")

## Answer key extraction (LLM-based, generic)

In [ ]:
def extract_answer_key_page(page_b64: str, page_number: int, total_pages: int) -> dict:
    """Extract everything on one answer-key page using taxonomy NAMES (not codes)."""
    prompt = f"""You are extracting an official EXAM ANSWER KEY. This is page {page_number} of {total_pages}.

The answer key contains BOTH the question text AND the correct answers (and marks per question).
Extract everything on this page exactly as written.

ANSWER TYPE TAXONOMY — classify each sub-question using NAMES, not numbers:

Top-level categories (use the exact name as "answer_type"):
  - "Selection-Based"
  - "Recall and Short Answer"
  - "Mathematical and Computational"
  - "Code and Programming"
  - "Diagrammatic and Visual"
  - "Table and Structured Data"
  - "Algorithm Execution and Tracing"
  - "Analysis and Interpretation"
  - "Design and Open-Ended"
  - "Essay and Explanation"

Subtypes (use the exact subtype name as "answer_subtype"):
{TAXONOMY_PROMPT_TEXT}

Return ONLY valid JSON (no markdown fences):
{{
  "exam_info": {{
    "course_code": "...",
    "course_name": "...",
    "date": "...",
    "total_marks": null,
    "outcomes": [
      {{"outcome_number": 1, "total_marks": null, "description": "..."}}
    ]
  }},
  "questions_on_page": [
    {{
      "question_number": "Q1",
      "sub_id": "a",
      "outcome_number": 1,
      "question_text": "exact question text as written",
      "marks": 2,
      "answer_type": "Diagrammatic and Visual",
      "answer_subtype": "Network Diagram (AOA/AON)",
      "correct_answer": "exact correct answer as written in the key",
      "mcq_options": {{"a": "...", "b": "...", "c": "...", "d": "..."}},
      "mcq_correct_option": "a",
      "has_drawing": false,
      "drawing_description": null
    }}
  ]
}}

RULES:
1. Fill "exam_info" ONLY on page 1. On later pages, set "exam_info": null.
2. For each question or sub-question visible on this page, emit ONE object in "questions_on_page".
   Sub-parts (a, b, c...) → same question_number, different sub_id.
   No sub-parts → sub_id = "a".

QUESTION NUMBER RULE (CRITICAL — read carefully):
   A question number is ONLY a code or label that appears BEFORE the question text.
   Valid forms include:
     - "Q1.", "Q1", "Question 1", "Question 3:"
     - Bare numbers: "1.", "1)", "(1)", "2.", "3."
     - Letters: "(a)", "(b)", "a)"
   Use it exactly as written (e.g. if the document shows "1.", return question_number = "Q1"
   — always add the "Q" prefix so bare numbers become proper IDs like "Q1", "Q2", "Q3").

   IMPORTANT: numbering often RESTARTS within each outcome. "1.", "2.", "3." under
   OUTCOME 2, then "1.", "2." again under OUTCOME 4 is normal. This is fine — the
   caller uses (outcome_number, question_number) together as the unique key, so the
   same question_number under different outcomes does not collide.

   Any code that appears AFTER the question text is NEVER a question number.
   It is always either:
     (a) the marks allocation — e.g. "/3 marks", "(6 Marks)", "3 marks", "out of 4"
         → put the number into the "marks" field.
     (b) the instructor's Study Outcome or internal category code — e.g. "[SO 1.1]",
         "[SO 2.2]", "2.3", "4.1" written at the end of the line
         → IGNORE it completely. Do NOT use it as question_number. Do NOT put it in
         answer_subtype. These codes refer to the instructor's grading rubric, not the
         answer type taxonomy.

   Only if NO label at all appears before the question text, generate a synthetic unique
   ID of the form "O{{outcome_number}}-Q{{n}}" where n is the sequential position of the
   question within that outcome (e.g. "O2-Q1", "O2-Q2").

3. "marks" = marks allocated to THIS sub-question as a NUMBER (integer or decimal, e.g. 2, 2.5, 0.5), or null if not shown.
4. "answer_type" = the top-level category NAME (e.g. "Diagrammatic and Visual").
5. "answer_subtype" = the specific subtype NAME (e.g. "Network Diagram (AOA/AON)").
6. If answer_type = "Diagrammatic and Visual", set has_drawing = true and write a detailed
   drawing_description (nodes, arrows, labels, durations, shapes, colors, annotations).
7. For answer_type = "Selection-Based": fill mcq_options with all choices and
   mcq_correct_option with the correct letter. For other types, set both to null.
8. Transcribe any non-Latin script (Arabic, etc.) in its original script. Preserve formulas exactly.
9. If the page has no questions, return "questions_on_page": [].
10. Return ONLY valid JSON. No text outside the JSON object. No trailing commas.
"""
    response = call_claude_with_retry(
        model=MODEL, max_tokens=4000, temperature=0.0,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": page_b64}},
            {"type": "text", "text": prompt}
        ]}]
    )
    text = response.content[0].text.strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            return json.loads(repair_json(text))
        except json.JSONDecodeError:
            print(f"    ⚠️  Page {page_number}: JSON parse failed.")
            return {"page_number": page_number, "raw_response": text, "parse_error": True}


def extract_answer_key(answer_key_path: Path) -> dict:
    """Extract full answer key from PDF or DOCX via LLM (no hardcoded content)."""
    print(f"📘 Extracting answer key from: {answer_key_path.name}")

    pages = answer_key_to_images(answer_key_path, dpi=200)
    print(f"   📄 {len(pages)} page(s) loaded")

    ak_drawings_dir = OUTPUT_DIR / "answer_key_drawings"
    ak_drawings_dir.mkdir(parents=True, exist_ok=True)

    exam_info = None
    all_sub_questions = []

    for i, page_img in enumerate(pages):
        page_num = i + 1
        print(f"   🔄 Page {page_num}...")
        page_b64 = image_to_base64(page_img)
        page_data = extract_answer_key_page(page_b64, page_num, len(pages))

        if page_data.get("parse_error"):
            continue

        if page_data.get("exam_info") and exam_info is None:
            exam_info = page_data["exam_info"]

        for sq in page_data.get("questions_on_page", []):
            sq["source_page"] = page_num

            if sq.get("answer_type") == "Diagrammatic and Visual" or sq.get("has_drawing"):
                qnum = sq.get("question_number", "Q?").replace(".", "_")
                sub  = sq.get("sub_id", "a")
                img_name = f"answer_key_{qnum}{sub}_page{page_num}.png"
                img_path = ak_drawings_dir / img_name
                page_img.save(str(img_path), "PNG")
                sq["answer_key_image_path"] = str(img_path)
                sq["has_drawing"] = True
                print(f"      🎨 Drawing reference saved: {img_name}")

            all_sub_questions.append(sq)

    # Renumber any synthetic IDs (O{n}-Q{m}) globally per outcome so that IDs
    # on different pages under the same outcome don't collide.
    # Non-synthetic IDs (e.g. "Q1", "Q2") are left untouched.
    import re as _re
    synthetic_counters = {}  # outcome_number -> running counter
    for sq in all_sub_questions:
        qnum = sq.get("question_number", "")
        onum = sq.get("outcome_number") or 1
        if _re.match(r"^O\d+-Q\d+$", str(qnum)):
            synthetic_counters[onum] = synthetic_counters.get(onum, 0) + 1
            sq["question_number"] = f"O{onum}-Q{synthetic_counters[onum]}"

    # Regroup: outcomes → questions → sub_questions
    outcomes_map = {}
    for sq in all_sub_questions:
        onum = sq.get("outcome_number") or 1
        qnum = sq.get("question_number", "Q?")
        if onum not in outcomes_map:
            outcomes_map[onum] = {"questions": {}}
        if qnum not in outcomes_map[onum]["questions"]:
            outcomes_map[onum]["questions"][qnum] = {
                "question_number": qnum,
                "question_text":   sq.get("question_text", ""),
                "sub_questions":   []
            }
        outcomes_map[onum]["questions"][qnum]["sub_questions"].append({
            "sub_id":                sq.get("sub_id", "a"),
            "sub_question_text":     sq.get("question_text", ""),
            "marks":                 sq.get("marks"),
            "answer_type":           sq.get("answer_type", ""),
            "answer_subtype":        sq.get("answer_subtype", ""),
            "correct_answer":        sq.get("correct_answer", ""),
            "mcq_options":           sq.get("mcq_options"),
            "mcq_correct_option":    sq.get("mcq_correct_option"),
            "has_drawing":           sq.get("has_drawing", False),
            "drawing_description":   sq.get("drawing_description"),
            "answer_key_image_path": sq.get("answer_key_image_path"),
            "source_page":           sq.get("source_page"),
        })

    outcomes_meta = (exam_info or {}).get("outcomes", []) if exam_info else []
    outcomes_list = []
    for onum in sorted(outcomes_map.keys()):
        meta = next((o for o in outcomes_meta if o.get("outcome_number") == onum), {})
        questions_list = list(outcomes_map[onum]["questions"].values())
        total = meta.get("total_marks")
        if total is None:
            total = sum((sq.get("marks") or 0)
                        for q in questions_list for sq in q["sub_questions"])
        outcomes_list.append({
            "outcome_number": onum,
            "total_marks":    total,
            "description":    meta.get("description"),
            "questions":      questions_list,
        })

    answer_key = {
        "source_file": answer_key_path.name,
        "exam_info": {
            "course_code":  (exam_info or {}).get("course_code"),
            "course_name":  (exam_info or {}).get("course_name"),
            "date":         (exam_info or {}).get("date"),
            "total_marks":  (exam_info or {}).get("total_marks"),
        },
        "outcomes": outcomes_list,
    }

    n_q = sum(len(o['questions']) for o in outcomes_list)
    n_sq = sum(len(sq['sub_questions']) for o in outcomes_list for sq in o['questions'])
    print(f"✅ Answer key extracted: {len(outcomes_list)} outcome(s), {n_q} question(s), {n_sq} sub-question(s)")
    return answer_key

print("✅ Answer key extractor ready.")

## Run answer key extraction

In [ ]:
answer_key = extract_answer_key(ANSWER_KEY_PATH)

answer_key_path = OUTPUT_DIR / "answer_key_structured.json"
with open(answer_key_path, 'w', encoding='utf-8') as f:
    json.dump(answer_key, f, indent=2, ensure_ascii=False)
print(f"💾 Answer key saved to {answer_key_path}")

# Summary table
rows = []
for outcome in answer_key['outcomes']:
    for q in outcome['questions']:
        for sq in q.get('sub_questions', []):
            rows.append({
                'Outcome': outcome['outcome_number'],
                'Question': q['question_number'],
                'Sub': sq['sub_id'],
                'Marks': sq['marks'],
                'Type': sq['answer_type'],
                'Subtype': sq['answer_subtype'],
                'Answer Preview': (sq['correct_answer'] or '')[:60] + '...'
            })
df_key = pd.DataFrame(rows)
print("\n📊 Exam Structure Summary:")
df_key

## Student info extraction (cover page)

In [ ]:
def extract_student_info(page_b64: str) -> dict:
    response = call_claude_with_retry(
        model=MODEL, max_tokens=600, temperature=0.1,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": page_b64}},
            {"type": "text", "text": """This is the cover page of a university exam paper. Extract the following information.
The student name may be in Arabic or English handwriting.
Also extract the marks written by the examiner.

Exams may have ANY number of learning outcomes (1, 2, 3, 4, or more). Capture marks for
every outcome you see on the page, keyed by the outcome number shown.

Return ONLY valid JSON (no markdown fences):
{
  "student_name": "...",
  "student_id": "...",
  "section": "...",
  "outcome_marks": {"1": null, "2": null},
  "total_marks": null
}

- "outcome_marks" is a dict: outcome number (as string) → marks obtained (number or null).
  Include every outcome/section visible on the cover page. If the exam labels sections
  differently (e.g. "Part A", "Section 1"), use those labels as keys instead.
- Marks may be integers OR decimals (e.g. 2, 2.5, 4.75). Return them as numbers, not strings.
- If a field is not readable, put null. Transcribe the name exactly as written."""}
        ]}]
    )
    text = response.content[0].text.strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"student_name": None, "student_id": None, "raw_response": text}

print("✅ Student info extractor ready.")

## Student answer extraction (transcribes only — types come from answer key)

In [ ]:
def extract_answers_from_page(page_b64: str, page_number: int, exam_structure: dict) -> dict:
    """Transcribe student answers on a single page. Does NOT classify answer types —
    those are stamped later from the answer key (source of truth)."""

    exam_info = exam_structure.get("exam_info", {}) or {}
    course_code = exam_info.get("course_code") or "this exam"
    course_name = exam_info.get("course_name") or ""
    course_label = f"{course_code} ({course_name})" if course_name else course_code

    # Expected questions list — gives the model context for ID matching
    expected_lines = []
    for outcome in exam_structure.get("outcomes", []):
        onum = outcome.get("outcome_number", "?")
        for q in outcome.get("questions", []):
            qnum = q.get("question_number", "Q?")
            sub_qs = q.get("sub_questions", [])
            has_multiple = len(sub_qs) > 1 or (sub_qs and sub_qs[0].get("sub_id", "a") != "a")
            for sq in sub_qs:
                sid = sq.get("sub_id", "a")
                qid = f"{qnum}{sid}" if has_multiple else qnum
                qtext = (sq.get("sub_question_text") or "")[:90]
                atype = sq.get("answer_type", "")
                marks = sq.get("marks", "?")
                expected_lines.append(f"  - {qid} [Outcome {onum}, {atype}, {marks} marks]: {qtext}")
    expected_block = "\n".join(expected_lines) if expected_lines else "  (no answer key structure available)"

    prompt = f"""You are an expert at reading handwritten exam answers. This is page {page_number} of a student's exam paper for {course_label}.

The exam contains the following questions overall. They may appear on any page of the student's paper — use this list to match each answer you find to the correct question_id:

{expected_block}

INK COLOR RULE — read before anything else:
  Student answers are written in the STUDENT'S pen, almost always BLUE or BLACK.
  Examiner grading marks are in RED or PINK ink.
  Before interpreting any mark on the page, decide whose mark it is:
    - Blue/black pen → the student. Goes into student_answer or student_selected_option.
    - Red/pink pen → the examiner. Goes into examiner_marks_raw / examiner_marks_numeric / examiner_comments.
  Examiner ink NEVER indicates the student's selection, NEVER adds to the student's answer,
  and NEVER changes which option the student chose. Examiner ticks (✓) and crosses (X) in
  red are judgments about the student's work — they are not part of the student's work.

CRITICAL TRANSCRIPTION RULES:
1. Write the student answer EXACTLY as the student wrote it — every word in English or any other
   language (Arabic, etc.). Preserve scripts. Do NOT replace Arabic with "[Arabic text]".
2. Keep ALL misspellings, grammatical errors, and abbreviations exactly as written.
   For student strikethroughs (student crossing out their own words), either omit the struck text
   or include it inline with a clear "[crossed out]" marker — but never include examiner strikethroughs.
3. If text is unclear, include your best guess with [unclear] markers.

ONE ENTRY PER SUB-QUESTION — NEVER MERGE:
   The expected-questions list above enumerates every sub-question (Q1a, Q1b, Q4e, Q4f, ...).
   Count the sub-question labels (a, b, c, d, e, f, 1, 2, 3, ...) printed ON THE PAGE and
   emit EXACTLY ONE JSON object per printed label. Each label owns its own student_answer.

   NEVER concatenate two sub-questions into one. Two short answers stacked near each other
   are still two separate answers and belong to two separate JSON objects.

   Bad example (DO NOT DO THIS):
     Page shows  e) "1.6 SR"  and  f) "Positiv 1.333"
     Wrong output: one object with question_id="Q4e", student_answer="1.6 SR Positiv 1.333"
   Correct output:
     {{"question_id": "Q4e", "student_answer": "1.6 SR", ...}}
     {{"question_id": "Q4f", "student_answer": "Positiv 1.333", ...}}

   If a sub-question's area is truly blank (no answer written at all), still emit an entry
   for it with student_answer = null. Do NOT skip labels you can see on the page.

DO NOT GUESS THE ANSWER FROM QUESTION MEANING:
   Your job is to REPORT what is physically on the page, not to reason about what the correct
   answer logically should be. If the question text sounds like it points to "Neutral" but the
   student circled "Resistant", report "Resistant". If the student left it blank, report blank.
   Never fabricate an answer from the question wording.

MCQ / SELECTION-BASED RULES — follow this procedure exactly:

   Step 1 — For each option (a, b, c, d, …), look at what STUDENT ink (blue/black) is on or
   around that option letter. Categorize each option into exactly ONE of these buckets:

     SELECTED — the student picked this option. Signs:
       • a circle or loop around the letter or the option
       • a tick / checkmark next to the letter or option
       • the letter written on the answer blank line
       • an arrow pointing into the option

     CROSSED OUT — the student eliminated this option. Signs:
       • an X through the letter or option
       • a slash or line through the option
       • a strikethrough over the option text

     UNMARKED — no student ink touches this option.

   Step 2 — Fill the "mcq_option_marks" field: a dict mapping each option letter to one
   of the three bucket names: "SELECTED", "CROSSED_OUT", or "UNMARKED". You MUST
   return this field for every MCQ.

   Step 3 — Derive student_selected_option from the dict using these rules IN ORDER:
     (a) If exactly ONE option is SELECTED → that letter.
     (b) If multiple options are SELECTED → the clearest/largest mark wins; note ambiguity
         in examiner_comments.
     (c) If ZERO options are SELECTED → student_selected_option = null.
         A CROSSED_OUT option is NEVER the selection.
         An UNMARKED option is NEVER the selection.
         Do NOT infer the answer from the question's wording.

   IMPORTANT distinctions:
     • Red/pink ink is the EXAMINER. It never indicates the student's selection.
       Examiner X, examiner ticks, and examiner red marks must not influence any bucket above.
     • A blank MCQ with no student marks produces null, not a guess.

   Worked example — exactly how to behave:
     Question: "The stakeholder is aware, yet neither supportive nor resistant…"
     Options on page: a) Unaware   b) Neutral   c) Resistant   d) Leading
     You observe in student's BLUE pen: blue X through 'a', small blue circle around 'c',
     small blue x near 'd', nothing on 'b'. In RED pen: examiner X under option d.
     → mcq_option_marks = {{"a": "CROSSED_OUT", "b": "UNMARKED",
                           "c": "SELECTED", "d": "CROSSED_OUT"}}
     → student_selected_option = "c"
     → student_answer = "c) Resistant"
     The examiner's red X is irrelevant for student_selected_option.
     The fact that "Neutral" sounds like the right answer is IRRELEVANT. The student
     circled c), so the answer is c).

   Output fields (required for every MCQ, set to null for non-MCQ questions):
     - "mcq_option_marks": dict {{letter → "SELECTED" | "CROSSED_OUT" | "UNMARKED"}}
     - "student_selected_option": single letter or null
     - "student_answer": the full text of the selected option as printed (e.g. "c) Resistant"),
       or null if nothing was selected.

DRAWING RULES:
   If the student drew any diagram, figure, flowchart, network, chart, table-of-boxes, or sketch,
   set has_drawing = true and write a DETAILED drawing_description covering:
     - All nodes/boxes/circles and their labels or numbers
     - All edges/arrows/lines, including direction and any labels (durations, activity names, etc.)
     - Any text the student wrote inside or next to the drawing
     - Any colors or examiner red-ink annotations around the drawing (note separately that those
       are examiner marks)

   A drawing_description like "network diagram drawn" is NOT acceptable — it conveys nothing.
   A grader reading only the description (without seeing the image) must be able to reconstruct
   the essential content.

   Bad example (DO NOT DO THIS):
     "Network diagram drawn with nodes 1-6 connected by activities A-G with durations"
   Good example (this level of detail is required):
     "AOA network: Node 1 (start) → Node 2 via A (duration 2). From Node 2: branch to Node 3
     via B (5), to Node 4 via C (1), to Node 5 via D (3). Node 3 → Node 6 via E (1). Node 4 →
     Node 6 via F (4). Node 5 → Node 6 via G (4). Node 6 labeled 'End'. Early/late times
     written above some nodes: [0/0] at 1, [2/2] at 2, [7/8] at 3, [3/5] at 4, [5/5] at 5,
     [9/9] at 6. Student labeled 'max' and 'min' arrows above the diagram."

CALCULATION RULES:
   Write formulas and numbers exactly as the student wrote them, even if wrong. Preserve the
   order of steps. Do not simplify or correct the student's math.

EXAMINER MARKS (red/pink ink) — return TWO fields per sub-question:
     - "examiner_marks_numeric": the number only (e.g. 4, 2.5, 0). Use null if only a tick/cross
       is present, or if nothing was written. Use 0 if the examiner wrote "0".
     - "examiner_marks_raw": the full notation (e.g. "4 (red ink)", "✓", "X", "2.5 circled"),
       or null if nothing was written.

DO NOT classify the answer type — that will be set separately from the answer key.

CRITICAL JSON FORMATTING RULES:
- Return ONLY valid JSON. No text outside the JSON object.
- ALL string values must be fully enclosed in double quotes.
- NEVER place descriptive text outside the quotes.
  WRONG:   "examiner_marks_raw": "9" in red ink
  CORRECT: "examiner_marks_raw": "9 (red ink)"
- If a field has no value, use null (without quotes).
- Do NOT use trailing commas.

Return ONLY valid JSON (no markdown fences):
{{
  "page_number": {page_number},
  "questions_on_page": [
    {{
      "question_id": "Q1a",
      "student_answer": "exact transcription preserved",
      "mcq_option_marks": null,
      "student_selected_option": null,
      "has_drawing": false,
      "drawing_description": null,
      "examiner_marks_numeric": null,
      "examiner_marks_raw": null,
      "examiner_comments": null
    }}
  ],
  "examiner_page_marks": null
}}

For MCQ questions specifically, mcq_option_marks MUST be filled as a dict. Example for an MCQ
with options a/b/c/d where the student crossed out a, circled c, and made no mark on b or d:
  "mcq_option_marks": {{"a": "CROSSED_OUT", "b": "UNMARKED",
                        "c": "SELECTED", "d": "UNMARKED"}}
For non-MCQ questions (short answer, calculation, drawing, essay), set mcq_option_marks to null.

Match each answer to the question_id from the expected list above. Be thorough — extract EVERY answer visible on this page."""

    response = call_claude_with_retry(
        model=MODEL, max_tokens=4000, temperature=0.1,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": page_b64}},
            {"type": "text", "text": prompt}
        ]}]
    )
    text = response.content[0].text.strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print(f"    ⚠️  JSON parse failed on page {page_number}, attempting repair...")
        try:
            return json.loads(repair_json(text))
        except json.JSONDecodeError:
            print(f"    ⚠️  Regex repair failed, asking LLM to fix JSON...")
            try:
                fix_response = call_claude_with_retry(
                    model=MODEL, max_tokens=3000, temperature=0.0,
                    messages=[{"role": "user", "content":
                        f"The following JSON is malformed. Fix it to be valid JSON. Return ONLY the fixed JSON, no explanation, no markdown fences.\n\n{text}"}]
                )
                fixed = fix_response.content[0].text.strip()
                fixed = re.sub(r'^```(?:json)?\s*|\s*```$', '', fixed)
                return json.loads(fixed)
            except Exception:
                print(f"    ❌ All repair attempts failed for page {page_number}")
                return {"page_number": page_number, "raw_response": text, "parse_error": True}

print("✅ Student answer extractor ready (transcription only).")

## Process student paper (stamps types from answer key + saves drawing pages)

In [ ]:
def process_student_paper(pdf_path: Path, answer_key: dict) -> dict:
    """Full pipeline:
       PDF → images → student info + answer transcription → stamp types from key → save drawing pages."""
    student_name_tag = pdf_path.stem
    print(f"\n{'='*60}")
    print(f"📄 Processing: {pdf_path.name}")
    print(f"{'='*60}")

    # 1. PDF → images
    print("  🔄 Converting PDF to images...")
    page_images = pdf_to_images(pdf_path, dpi=200)
    print(f"  ✅ {len(page_images)} pages converted")

    # 2. Cover page → student info
    print("  🔄 Extracting student info from cover page...")
    cover_b64 = image_to_base64(page_images[0])
    student_info = extract_student_info(cover_b64)
    print(f"  ✅ Student: {student_info.get('student_name', 'Unknown')}")
    print(f"     ID: {student_info.get('student_id', 'Unknown')}")

    # 3. Extract answers from each subsequent page
    all_answers = []
    for page_idx in range(1, len(page_images)):
        page_num = page_idx + 1
        print(f"  🔄 Extracting answers from page {page_num}...")
        page_b64 = image_to_base64(page_images[page_idx])
        page_answers = extract_answers_from_page(page_b64, page_num, answer_key)
        all_answers.append(page_answers)
        if 'questions_on_page' in page_answers:
            n = len(page_answers['questions_on_page'])
            print(f"  ✅ Page {page_num}: {n} answers extracted")
        else:
            print(f"  ⚠️ Page {page_num}: parsing issue, raw response saved")

    # 4. Stamp answer_type / subtype / marks / outcome from the answer key
    key_lookup = {}
    for outcome in answer_key.get("outcomes", []):
        for q in outcome.get("questions", []):
            qnum = q.get("question_number", "")
            sub_qs = q.get("sub_questions", [])
            has_multiple = len(sub_qs) > 1 or (sub_qs and sub_qs[0].get("sub_id", "a") != "a")
            for sq in sub_qs:
                sid = sq.get("sub_id", "a")
                qid = f"{qnum}{sid}" if has_multiple else qnum
                key_lookup[qid.upper()] = {
                    "answer_type":          sq.get("answer_type", ""),
                    "answer_subtype":       sq.get("answer_subtype", ""),
                    "expected_marks":       sq.get("marks"),
                    "outcome_number":       sq.get("outcome_number") if "outcome_number" in sq else outcome.get("outcome_number"),
                    "correct_answer":       sq.get("correct_answer", ""),
                    "mcq_key_correct_option": sq.get("mcq_correct_option"),
                    "answer_key_image_path": sq.get("answer_key_image_path"),
                }

    for page_data in all_answers:
        if "questions_on_page" not in page_data:
            continue
        for q in page_data["questions_on_page"]:
            qid = (q.get("question_id") or "").upper()
            info = key_lookup.get(qid)
            if info:
                q["answer_type"]           = info["answer_type"]
                q["answer_subtype"]        = info["answer_subtype"]
                q["expected_marks"]        = info["expected_marks"]
                q["outcome_number"]        = info["outcome_number"]
                q["correct_answer"]        = info["correct_answer"]
                q["mcq_key_correct_option"] = info["mcq_key_correct_option"]
                q["answer_key_image_path"] = info["answer_key_image_path"]
            else:
                q["answer_type"]           = ""
                q["answer_subtype"]        = ""
                q["expected_marks"]        = None
                q["outcome_number"]        = None
                q["unmatched_question"]   = True

    # 5. For drawing-type questions: save full pages (from question's page up to next question's page)
    drawing_paths = []
    question_locations = []  # ordered list of (page_num, question_dict)
    for page_data in all_answers:
        if 'questions_on_page' not in page_data:
            continue
        pnum = page_data.get('page_number')
        for q in page_data['questions_on_page']:
            question_locations.append((pnum, q))

    for i, (pnum, q) in enumerate(question_locations):
        is_drawing_q = q.get('answer_type') == "Diagrammatic and Visual" or q.get('has_drawing') is True
        if not is_drawing_q:
            continue

        qid = q.get('question_id', 'unknown').replace('.', '_')

        # Find the page the NEXT question starts on. We save pages from this
        # drawing question's page up to (but NOT including) that page — so if the
        # next question is on the same page, we only save this page, and we only
        # bleed into later pages when they're actually blank-overflow pages.
        next_q_page = None
        for (npnum, _) in question_locations[i+1:]:
            if npnum > pnum:
                next_q_page = npnum
                break
        last_page_to_save = (next_q_page - 1) if next_q_page else len(page_images)

        saved_for_this_q = []
        for search_pn in range(pnum, last_page_to_save + 1):
            idx = search_pn - 1
            if not (0 <= idx < len(page_images)):
                continue
            draw_filename = f"{student_name_tag}_{qid}_page{search_pn}.png"
            draw_path = DRAWINGS_DIR / draw_filename
            page_images[idx].save(str(draw_path), "PNG")
            saved_for_this_q.append(str(draw_path))
            drawing_paths.append(str(draw_path))

        if saved_for_this_q:
            q['drawing_file_paths'] = saved_for_this_q
            q['has_drawing'] = True
            print(f"  🎨 {qid}: saved {len(saved_for_this_q)} full page(s)")

    return {
        "file_name":      pdf_path.name,
        "student_info":   student_info,
        "pages":          all_answers,
        "drawing_files":  drawing_paths,
        "processed_at":   datetime.now().isoformat(),
    }

print("✅ Processing pipeline ready.")

## Run the pipeline on all students

In [ ]:
all_results = {}

for pdf_path in sorted(student_paths):
    out_file = OUTPUT_DIR / f"{pdf_path.stem}_extracted.json"
    if out_file.exists():
        print(f"⏭️  Skipping {pdf_path.name} (already processed)")
        with open(out_file, encoding="utf-8") as f:
            all_results[pdf_path.stem] = json.load(f)
        continue

    result = process_student_paper(pdf_path, answer_key)
    student_tag = pdf_path.stem
    all_results[student_tag] = result

    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"  💾 Results saved: {out_file.name}")

combined_path = OUTPUT_DIR / "all_students_extracted.json"
with open(combined_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print(f"✅ ALL {len(all_results)} STUDENTS PROCESSED")
print(f"💾 Combined results: {combined_path}")
print(f"🎨 Drawings saved in: {DRAWINGS_DIR}")
print(f"{'='*60}")

## Flatten to CSV

In [ ]:
def flatten_student_answers(result: dict) -> list:
    student_name = result['student_info'].get('student_name', 'Unknown')
    student_id   = result['student_info'].get('student_id', 'Unknown')
    rows = []
    for page_data in result['pages']:
        if 'questions_on_page' not in page_data:
            continue
        for q in page_data['questions_on_page']:
            rows.append({
                'Student': student_name,
                'ID': student_id,
                'File': result['file_name'],
                'Outcome': q.get('outcome_number', ''),
                'Question': q.get('question_id', ''),
                'Answer_Type': q.get('answer_type', ''),
                'Answer_Subtype': q.get('answer_subtype', ''),
                'Expected_Marks': q.get('expected_marks', ''),
                'Student_Answer': q.get('student_answer', ''),
                'Student_Selected_Option': q.get('student_selected_option', ''),
                'MCQ_Option_Marks': json.dumps(q.get('mcq_option_marks'), ensure_ascii=False) if q.get('mcq_option_marks') else '',
                'Correct_Answer': q.get('correct_answer', ''),
                'MCQ_Key_Correct_Option': q.get('mcq_key_correct_option', ''),
                'Has_Drawing': q.get('has_drawing', False),
                'Drawing_Description': q.get('drawing_description', ''),
                'Drawing_File_Paths': '; '.join(q.get('drawing_file_paths', []) or []),
                'Examiner_Marks_Numeric': q.get('examiner_marks_numeric', ''),
                'Examiner_Marks_Raw': q.get('examiner_marks_raw', ''),
                'Examiner_Comments': q.get('examiner_comments', ''),
            })
    return rows

all_rows = []
for tag, result in all_results.items():
    all_rows.extend(flatten_student_answers(result))

df_answers = pd.DataFrame(all_rows)
csv_path = OUTPUT_DIR / "all_answers.csv"
df_answers.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"📊 CSV saved: {csv_path} ({len(df_answers)} rows)")
df_answers

## Generate report + download

In [ ]:
def build_comprehensive_report(all_results: dict, answer_key: dict) -> str:
    info = answer_key.get('exam_info', {}) or {}
    lines = [
        "# Exam Extraction Report",
        f"**Course:** {info.get('course_code') or '?'} — {info.get('course_name') or ''}",
        f"**Date:** {info.get('date') or ''}",
        f"**Total Marks:** {info.get('total_marks') or ''}",
        f"**Students Processed:** {len(all_results)}",
        f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        "", "---", ""
    ]
    for tag, result in all_results.items():
        sinfo = result['student_info']
        lines.append(f"## Student: {sinfo.get('student_name', 'Unknown')}")
        lines.append(f"- **ID:** {sinfo.get('student_id', 'N/A')}")
        lines.append(f"- **File:** {result['file_name']}")
        # Print every outcome's marks dynamically (handles 1, 2, 3, ... outcomes)
        outcome_marks = sinfo.get('outcome_marks') or {}
        if outcome_marks:
            for onum, omark in sorted(outcome_marks.items(), key=lambda x: str(x[0])):
                lines.append(f"- **Outcome {onum}:** {omark if omark is not None else '?'}")
        lines.append(f"- **Total:** {sinfo.get('total_marks', '?')}")
        lines.append("")
        for page_data in result['pages']:
            if 'questions_on_page' not in page_data:
                continue
            for q in page_data['questions_on_page']:
                qid = q.get('question_id', '?')
                atype = q.get('answer_type', '')
                asub = q.get('answer_subtype', '')
                marks_raw = q.get('examiner_marks_raw', '')
                marks_num = q.get('examiner_marks_numeric', '')
                emarks = q.get('expected_marks', '')
                marks_display = marks_raw if marks_raw else (str(marks_num) if marks_num not in (None, '') else '—')
                lines.append(f"### {qid} [{atype} / {asub}] (Marks: {marks_display} of {emarks})")
                if q.get('has_drawing'):
                    lines.append(f"🎨 **Drawing:** {q.get('drawing_description', 'See saved image')}")
                lines.append(f"**Student Answer:** {q.get('student_answer', 'N/A')}")
                lines.append("")
        if result.get('drawing_files'):
            lines.append("**Drawing files:**")
            for df_ in result['drawing_files']:
                lines.append(f"- `{Path(df_).name}`")
            lines.append("")
        lines.append("---")
        lines.append("")
    return "\n".join(lines)

report = build_comprehensive_report(all_results, answer_key)
report_path = OUTPUT_DIR / "extraction_report.md"
report_path.write_text(report, encoding='utf-8')
print(f"📄 Report saved: {report_path}")

# Zip everything for download
import shutil
zip_path = WORK_DIR / "exam_extraction_results"
shutil.make_archive(str(zip_path), 'zip', str(OUTPUT_DIR))
print("\n📦 Download the results package:")
files.download(str(zip_path) + '.zip')